# PHYS 690 - Lecture 14: FFT Fundamentals and Periodic Field Solvers

## From time-angular-frequency analysis to a Fourier-space solution of Poisson's equation

A Fourier transform describes a sampled signal as a sum of oscillating basis functions. The **discrete Fourier transform** (DFT) is the finite-dimensional transformation appropriate for numerical data. The **fast Fourier transform** (FFT) is an efficient algorithm for calculating exactly those same DFT coefficients.

Part 1 uses the traditional conjugate variables time $t$ and angular frequency $\omega$, measured in radians per second. Part 2 then applies the same numerical transform to position $x$, where the conjugate variable is angular wave number $k$, measured in radians per unit length.

## Learning goals

By the end of class, you should be able to:

1. write the forward and inverse DFT for $N$ uniformly sampled time values;
2. distinguish the mathematical DFT from the FFT algorithm used to compute it;
3. explain why FFT work scales as approximately $\mathcal O(N\log N)$ rather than $\mathcal O(N^2)$;
4. map FFT array indices to angular frequencies $\omega$, amplitudes, and phases;
5. translate the time-$\omega$ workflow into the position-$k$ basis;
6. derive and implement the Fourier-space solution of a periodic Poisson equation; and
7. verify the computed field by comparing with a known solution.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.fft import rfft, irfft, rfftfreq

# As in Lectures 11 and 12, use normalized electrostatic units.
eps0 = 1.0

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})


## Part 1: From a sampled time signal to angular-frequency components

Suppose a periodic signal $y(t)$ is sampled $N$ times at a fixed sampling interval $\Delta t$:

$$
t_j=j\Delta t,
\qquad
j=0,1,\ldots,N-1.
$$

The sampling rate, angular sampling frequency, and total observation duration are

$$
f_s=\frac{1}{\Delta t},
\qquad
\omega_s=2\pi f_s=\frac{2\pi}{\Delta t},
\qquad
T=N\Delta t.
$$

For a periodic record, $t=T$ is not included because it duplicates $t=0$ in the next period. The sampled values form a vector

$$
\mathbf y=(y_0,y_1,\ldots,y_{N-1}),
\qquad
y_j=y(t_j).
$$

The DFT expresses this time-domain vector in an angular-frequency basis. No information is created: the $N$ time samples are replaced by $N$ complex coefficients that encode the amplitudes and phases of the angular frequencies distinguishable in the sampled record.

### The full DFT definition and its real-input form

Using the default `scipy.fft` convention, the forward discrete Fourier transform (DFT) is

$$
\boxed{Y_m=\sum_{j=0}^{N-1}y_j e^{-i2\pi jm/N}},
\qquad
m=0,1,\ldots,N-1,
$$

and the inverse transform is

$$
\boxed{y_j=\frac{1}{N}\sum_{m=0}^{N-1}Y_m e^{+i2\pi jm/N}}.
$$

The coefficient index $m$ corresponds to angular frequency

$$
\omega_m=2\pi\frac{m}{N\Delta t}=\frac{2\pi m}{T}.
$$

Euler's formula,

$$
e^{i\theta}=\cos\theta+i\sin\theta,
$$

shows why the coefficients are complex. The magnitude $|Y_m|$ measures the component strength, while $\arg(Y_m)$ gives its phase. The factor $1/N$ appears in the inverse transform under SciPy's default normalization.

Every signal and field in this lecture is real valued. Its coefficients obey conjugate symmetry,

$$
Y(-\omega)=Y(\omega)^*,
$$

so the negative-frequency coefficients contain no independent information. From this point onward, the code uses `rfft` to store only the nonnegative-frequency coefficients, `rfftfreq` to label them, and `irfft` to reconstruct real samples. These functions evaluate the same DFT defined above while avoiding the redundant half of the array.


### Example time-dependent signal

$$ y(t) = 1+0.75\cos(2\pi t)+0.50\sin(4\pi t)$$


In [ ]:
# Use eight samples over one second so every sample can be inspected clearly.
N_demo = 8
sample_rate_demo = 8.0  # samples/s
dt_demo = 1.0 / sample_rate_demo
duration_demo = N_demo * dt_demo
j_demo = np.arange(N_demo)
t_demo = j_demo * dt_demo

# Define angular frequencies for 1 and 2 cycles per second.
omega_1_demo = 2.0 * np.pi * 1.0  # rad/s
omega_2_demo = 2.0 * np.pi * 2.0  # rad/s

# Generate a signal with a mean and two angular-frequency components.
y_demo = (
    1.0
    + 0.75 * np.cos(omega_1_demo * t_demo)
    + 0.50 * np.sin(omega_2_demo * t_demo)
)

# Evaluate the generating function densely only for visualization.
# The FFT itself receives only the eight sampled values in y_demo.
t_dense = np.linspace(0.0, duration_demo, 600)
constant_component = np.ones_like(t_dense)
cosine_component = 0.75 * np.cos(omega_1_demo * t_dense)
sine_component = 0.50 * np.sin(omega_2_demo * t_dense)

demo_function_df = pd.DataFrame({
    "time_s": t_dense,
    "constant": constant_component,
    "cosine_omega_2pi": cosine_component,
    "sine_omega_4pi": sine_component,
    "total": constant_component + cosine_component + sine_component,
})
demo_samples_df = pd.DataFrame({"time_s": t_demo, "signal": y_demo})

# First show the continuous function and the eight values sampled from it.
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4), constrained_layout=True)
axes[0].plot(demo_function_df["time_s"], demo_function_df["total"],
             lw=2.5, label="full generating signal")
axes[0].scatter(demo_samples_df["time_s"], demo_samples_df["signal"],
                color="black", s=35, zorder=3, label="eight time samples")
axes[0].set_xlabel("Time t (s)")
axes[0].set_ylabel("Signal y(t)")
axes[0].set_title("Time-domain signal and samples")
axes[0].legend(fontsize=9)

# Overlay the individual angular-frequency components and their sum.
axes[1].plot(demo_function_df["time_s"], demo_function_df["constant"],
             "--", label="1")
axes[1].plot(demo_function_df["time_s"], demo_function_df["cosine_omega_2pi"],
             "--", label=r"$0.75\cos(2\pi t)$")
axes[1].plot(demo_function_df["time_s"], demo_function_df["sine_omega_4pi"],
             "--", label=r"$0.50\sin(4\pi t)$")
axes[1].plot(demo_function_df["time_s"], demo_function_df["total"],
             color="black", lw=2.5, label="sum")
axes[1].set_xlabel("Time t (s)")
axes[1].set_ylabel("Component value")
axes[1].set_title("Angular-frequency components sum to the signal")
axes[1].legend(fontsize=9, ncol=2)

fig.suptitle(r"$y(t)=1+0.75\cos(2\pi t)+0.50\sin(4\pi t)$")
plt.show()

### Now compute the angular-frequency spectrum

The numerical transform receives only the eight values in `y_demo`. Because those samples are real, `rfft` returns the independent coefficients from zero through the Nyquist angular frequency. We use `rfftfreq` to obtain frequencies in cycles per second, multiply by $2\pi$ to obtain angular frequencies in radians per second, and reconstruct the time samples with `irfft`.

For a one-sided amplitude spectrum, multiply $|Y_m|/N$ by two to account for the omitted negative-frequency partner. Do not double the zero-frequency or Nyquist coefficients because each occurs only once.


In [ ]:
# Transform the eight real samples and label the independent coefficients.
Y_demo = rfft(y_demo)
omega_demo = 2.0 * np.pi * rfftfreq(N_demo, d=dt_demo)

# Convert coefficient magnitudes to physical one-sided amplitudes.
amplitude_demo = 2.0 * np.abs(Y_demo) / N_demo
amplitude_demo[0] /= 2.0
amplitude_demo[-1] /= 2.0

# Return from the angular-frequency representation to the real time samples.
y_reconstructed = irfft(Y_demo)

# Store original and reconstructed samples together for the round-trip check.
dft_demo_df = pd.DataFrame({
    "time_s": t_demo,
    "signal": y_demo,
    "reconstructed_signal": y_reconstructed,
})

# Store the independent coefficients, their amplitudes, and their phases.
coefficient_demo_df = pd.DataFrame({
    "omega_rad_s": omega_demo,
    "real_Y_m": Y_demo.real,
    "imag_Y_m": Y_demo.imag,
    "amplitude": amplitude_demo,
    "phase_rad": np.angle(Y_demo),
})

# Phases are meaningful only for coefficients that are appreciably nonzero.
significant_demo_df = coefficient_demo_df[
    coefficient_demo_df["amplitude"] > 1.0e-12
]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)

# Verify that the inverse transform reproduces every sampled value.
axes[0].stem(dft_demo_df["time_s"], dft_demo_df["signal"],
             linefmt="tab:blue", markerfmt="o", basefmt=" ",
             label="original samples")
axes[0].plot(dft_demo_df["time_s"], dft_demo_df["reconstructed_signal"],
             "s--", color="tab:orange", label="irfft reconstruction")
axes[0].set_xlabel("Time t (s)")
axes[0].set_ylabel("Signal")
axes[0].set_title("Time-domain reconstruction")
axes[0].legend(fontsize=9)

# Plot only the nonnegative angular frequencies returned by rfft.
axes[1].stem(coefficient_demo_df["omega_rad_s"],
             coefficient_demo_df["amplitude"], basefmt=" ")
axes[1].set_xlabel(r"Angular frequency $\omega$ (rad/s)")
axes[1].set_ylabel("One-sided amplitude")
axes[1].set_title("Independent amplitudes")

# Show phase only where a physical component is present.
axes[2].stem(significant_demo_df["omega_rad_s"],
             significant_demo_df["phase_rad"], basefmt=" ")
axes[2].set_xlabel(r"Angular frequency $\omega$ (rad/s)")
axes[2].set_ylabel("Phase (rad)")
axes[2].set_title("Phases of occupied modes")

fig.suptitle("Eight-point real discrete Fourier transform")
plt.show()


### Reading the eight-point result

- The coefficient at $\omega=0$ has amplitude 1 and is the signal mean, or DC component.
- The coefficient at $\omega=2\pi$ rad/s has amplitude $0.75$ and phase zero, matching the cosine term.
- The coefficient at $\omega=4\pi$ rad/s has amplitude $0.50$ and phase $-\pi/2$, because $\sin(\omega t)=\cos(\omega t-\pi/2)$.
- Values near $10^{-16}$ are floating-point roundoff, not additional angular frequencies.

The omitted coefficients at negative angular frequency are fixed by $Y(-\omega)=Y(\omega)^*$. Thus the one-sided output contains all information needed to reconstruct the real signal.


### Why the FFT is fast

Calculating the DFT directly requires evaluating $N$ terms for each of $N$ Fourier coefficients. The amount of work therefore grows as

$$
\mathcal O(N^2).
$$

The FFT calculates the **same DFT coefficients** but reorganizes the sums so intermediate results can be reused. The commonly used Cooley-Tukey algorithm separates the samples into smaller transforms, combines their results, and repeats this process recursively. There are approximately $\log_2N$ combination stages, and each stage processes a total of $N$ values. Its computational cost therefore scales as

$$
\boxed{\mathcal O(N\log N)}.
$$

Big-$\mathcal O$ notation describes how the computational work grows for large $N$; it omits constant factors and lower-order terms. The important practical result is that $N\log N$ grows much more slowly than $N^2$, making large Fourier transforms feasible.

For a detailed derivation, see [Deriving the Fast Fourier Transform in Engineering LibreTexts](https://eng.libretexts.org/Bookshelves/Electrical_Engineering/Signal_Processing_and_Modeling/Signals_and_Systems_%28Baraniuk_et_al.%29/13%3A_Capstone_Signal_Processing_Topics/13.03%3A_Deriving_the_Fast_Fourier_Transform).

In [ ]:
# Compare operation-count scaling, not wall-clock timing on one computer.
transform_sizes = 2 ** np.arange(3, 21)
work_df = pd.DataFrame({
    "N": transform_sizes,
    "direct_DFT_scaling": transform_sizes**2,
    "FFT_scaling": transform_sizes * np.log2(transform_sizes),
})

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.loglog(work_df["N"], work_df["direct_DFT_scaling"],
          marker="o", label=r"direct DFT: $N^2$")
ax.loglog(work_df["N"], work_df["FFT_scaling"],
          marker="s", label=r"FFT: $N\log_2N$")
ax.set_xlabel("Number of samples N")
ax.set_ylabel("Relative operation count")
ax.set_title("Why the FFT matters for large numerical grids")
ax.legend()
plt.show()

### From coefficient indices to angular frequencies

`rfftfreq(N, d=dt)` returns the nonnegative ordinary frequencies in cycles per unit time that correspond to the coefficients from `rfft`. Multiply by $2\pi$ to obtain angular frequency:

$$
\omega_m=2\pi\,\texttt{rfftfreq}(N,\Delta t).
$$

The spacing between adjacent angular frequencies is

$$
\Delta\omega=\frac{2\pi}{T}.
$$

The **Nyquist angular frequency** is the highest angular frequency that can be represented uniquely with sample spacing $\Delta t$. It corresponds to two samples per oscillation:

$$
\omega_\mathrm{Nyquist}=\frac{\pi}{\Delta t}.
$$

Frequencies above this limit are indistinguishable from lower frequencies after sampling, a phenomenon called **aliasing**. For even $N$, `rfft` includes the Nyquist coefficient as its final entry.

For a real cosine $A\cos(\omega t+\delta)$, a one-sided amplitude spectrum uses $2|Y_m|/N$ to account for the conjugate coefficient that is not stored. The zero and Nyquist coefficients have no separate partners and are not doubled.

**Predict before running:** Which two positive angular frequencies should appear for the signal below? Which should have the larger amplitude? If one component is shifted in time, should its peak move, change amplitude, or change only its complex phase?


In [ ]:
# Sample two angular frequencies over a two-second time record.
sample_rate_basic = 64.0  # samples/s
duration_basic = 2.0  # s
N_basic = int(sample_rate_basic * duration_basic)
dt_basic = 1.0 / sample_rate_basic
time_basic = np.arange(N_basic) * dt_basic

omega_1_basic = 2.0 * np.pi * 3.0  # rad/s
omega_2_basic = 2.0 * np.pi * 7.0  # rad/s
signal_basic = (
    1.2 * np.cos(omega_1_basic * time_basic)
    + 0.4 * np.sin(omega_2_basic * time_basic)
)

signal_df = pd.DataFrame({"time_s": time_basic, "signal": signal_basic})

# Convert rfftfreq output from cycles/s to angular frequency in rad/s.
Y_basic = rfft(signal_df["signal"].to_numpy())
omega_basic = 2.0 * np.pi * rfftfreq(N_basic, d=dt_basic)
amplitude = 2.0 * np.abs(Y_basic) / N_basic
amplitude[0] /= 2.0
amplitude[-1] /= 2.0

spectrum_df = pd.DataFrame({
    "omega_rad_s": omega_basic,
    "amplitude": amplitude,
    "phase_rad": np.angle(Y_basic),
})

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
axes[0].plot(signal_df["time_s"], signal_df["signal"])
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Signal")
axes[0].set_title("Uniformly sampled time signal")

axes[1].stem(spectrum_df["omega_rad_s"], spectrum_df["amplitude"], basefmt=" ")
axes[1].set_xlim(0, 16.0 * np.pi)
axes[1].set_xlabel(r"Angular frequency $\omega$ (rad/s)")
axes[1].set_ylabel("One-sided amplitude")
axes[1].set_title("Angular-frequency spectrum from rfft")

fig.tight_layout()
plt.show()

### `scipy.fft` quick reference for real-valued data

Here $N$ is the number of evenly spaced real samples and $d$ is the sample spacing.

| Function | Input | Returns |
|---|---|---|
| `rfft(y)` | $N$ real samples | The independent complex coefficients at nonnegative frequencies |
| `irfft(Y)` | Coefficients returned by `rfft` | The reconstructed real samples |
| `rfftfreq(N, d)` | Sample count and spacing | Nonnegative frequencies in cycles per unit, ordered to match `rfft` |

`rfftfreq` returns cycles per unit coordinate. In Part 1, convert to angular frequency with

$$
\omega=2\pi\,\texttt{rfftfreq}(N,\Delta t).
$$

The same conversion gives angular wave number $k$ when $d=\Delta x$. See the [SciPy real FFT documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.fft.rfft.html) for details.


### In-class coding activity 1 - Analyze and reconstruct a time signal

Use `rfft`, `rfftfreq`, and `irfft` to transform the supplied time signal, label the coefficients by angular frequency $\omega$ in radians per second, identify its mean and two oscillating components, and reconstruct the original samples.

Remember that `rfftfreq` returns cycles per second. The conversion to the notation used here is $\omega=2\pi f$.

In [ ]:
# Define a four-second time record sampled 64 times per second.
sample_rate_activity = 64.0  # samples/s
duration_activity = 4.0  # s
N_activity = int(sample_rate_activity * duration_activity)
dt_activity = 1.0 / sample_rate_activity
time_activity = np.arange(N_activity) * dt_activity

# Define the two angular frequencies in rad/s.
omega_1_activity = 2.0 * np.pi * 4.0
omega_2_activity = 2.0 * np.pi * 9.0

# Construct a signal with a nonzero mean and two angular-frequency components.
activity_signal = (
    0.25
    + 1.10 * np.cos(omega_1_activity * time_activity + 0.40)
    + 0.55 * np.sin(omega_2_activity * time_activity)
)

activity_signal_df = pd.DataFrame({
    "time_s": time_activity,
    "signal": activity_signal,
})

# Plot the time-domain information available before the FFT.
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(activity_signal_df["time_s"], activity_signal_df["signal"], lw=1.2)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Sampled signal")
ax.set_title("Time signal to analyze with scipy.fft")
plt.show()

#### Step 1: Calculate and interpret the one-sided angular-frequency spectrum

`rfft` returns $N/2+1$ coefficients for an even-length real signal. Use `rfftfreq(N_activity, d=dt_activity)` and multiply its result by $2\pi$ to obtain $\omega$ in radians per second.

After calculating $2|Y_m|/N$, remember that the zero and Nyquist coefficients do not have separate negative-$\omega$ partners and should not be doubled.

In [ ]:
# Extract the time samples from the DataFrame for the numerical transform.
activity_samples = activity_signal_df["signal"].to_numpy()

# TODO: Compute the independent Fourier coefficients for this real signal.
activity_coefficients = ...

# TODO: Use rfftfreq and convert its output to angular frequency in rad/s.
activity_omega_rad_s = ...

# TODO: Calculate the one-sided amplitude 2*abs(Y_m)/N.
activity_amplitude = ...

# The zero-angular-frequency and Nyquist coefficients occur only once.
activity_amplitude[0] /= 2.0
activity_amplitude[-1] /= 2.0

activity_spectrum_df = pd.DataFrame({
    "omega_rad_s": activity_omega_rad_s,
    "amplitude": activity_amplitude,
    "phase_rad": np.angle(activity_coefficients),
})

# TODO: Select the three rows with the largest amplitudes.
dominant_omega_df = ...
display(dominant_omega_df.sort_values("omega_rad_s"))

# Plot the one-sided angular-frequency spectrum.
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.stem(activity_spectrum_df["omega_rad_s"],
        activity_spectrum_df["amplitude"], basefmt=" ")
ax.set_xlim(0, 22.0 * np.pi)
ax.set_xlabel(r"Angular frequency $\omega$ (rad/s)")
ax.set_ylabel("One-sided amplitude")
ax.set_title("Angular frequencies identified with rfft")
plt.show()

#### Step 2: Reconstruct the time samples

The coefficients are another representation of the same finite time record. Pass them to `irfft`; for this even-length signal, the original sample count is recovered automatically. A successful round trip should reproduce the input to floating-point precision.

After completing the cell, discuss how the mean, amplitude, and phase appear in $Y(\omega)$ and why reconstruction is a useful numerical check.

In [ ]:
# TODO: Apply the inverse real FFT to reconstruct the time samples.
activity_reconstructed = ...

# Add reconstructed values to the time-domain DataFrame for comparison.
activity_signal_df["reconstructed"] = activity_reconstructed

# TODO: Compute the maximum absolute reconstruction error over all samples.
max_reconstruction_error = ...
print(f"Maximum reconstruction error = {max_reconstruction_error:.3e}")

# Overlay the original time samples and inverse-transform result.
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(activity_signal_df["time_s"], activity_signal_df["signal"],
        lw=2.5, label="original samples")
ax.plot(activity_signal_df["time_s"], activity_signal_df["reconstructed"],
        "--", lw=1.8, label="inverse FFT")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Signal")
ax.set_title(r"Time-$\omega$ round trip")
ax.legend()
plt.show()

## Part 2: From the time-$\omega$ basis to the position-$k$ basis

Part 1 transformed between the conjugate variables time $t$ and angular frequency $\omega$. For uniformly spaced positions $x_j=j\Delta x$, the same DFT transforms between position $x$ and angular wave number $k$.

SciPy returns frequencies in cycles per unit coordinate, so both angular variables require a factor of $2\pi$:

$$
\omega=2\pi\,\texttt{rfftfreq}(N,\Delta t),
\qquad
k=2\pi\,\texttt{rfftfreq}(N,\Delta x).
$$

The correspondence is

$$
\begin{array}{c|c}
\text{time-angular-frequency basis} & \text{position-wave-number basis}\\
\hline
t & x\\
\Delta t & \Delta x\\
\omega & k
\end{array}
$$

Because the spatial charge, potential, and electric field are real valued, the code again stores only $k\geq0$. The unlisted negative-$k$ coefficients follow from conjugate symmetry.

### Differentiation mode by mode

A periodic spatial function has the Fourier representation

$$
g(x)=\sum_m \widetilde g_m e^{ik_mx}.
$$

Therefore,

$$
\frac{d}{dx}e^{ik_mx}=ik_me^{ik_mx},
\qquad
\frac{d^2}{dx^2}e^{ik_mx}=-k_m^2e^{ik_mx}.
$$

Spectral differentiation becomes

$$
g_j\xrightarrow{\mathrm{rfft}}\widetilde g_m
\xrightarrow{\times ik_m}ik_m\widetilde g_m
\xrightarrow{\mathrm{irfft}}\left(\frac{dg}{dx}\right)_j.
$$

Unlike a finite-difference stencil, this is a global operation. It can be extremely accurate for smooth periodic functions, while discontinuities produce Gibbs oscillations.

### Apply the position-$k$ representation to electrostatics

In SI units, a periodic one-dimensional charge density obeys

$$
-\frac{d^2V}{dx^2}=\frac{\rho(x)}{\epsilon_0},
\qquad
E(x)=-\frac{dV}{dx}.
$$

The equations will retain $\epsilon_0$ so their physical structure and dimensions remain visible. As in Lectures 11 and 12, however, the code uses **normalized units** and fixes `eps0 = 1.0`. The numerical coordinate $x$, charge density $\rho$, potential $V$, and field $E$ are therefore normalized quantities rather than SI values. A dimensional calculation would use the physical $\epsilon_0$ and consistent dimensional inputs.

Transforming Poisson's equation from $x$ to $k$ gives

$$
k_m^2\widetilde V_m=\frac{\widetilde\rho_m}{\epsilon_0}.
$$

For $k_m\ne0$,

$$
\boxed{\widetilde V_m=\frac{\widetilde\rho_m}{\epsilon_0k_m^2}},
\qquad
\boxed{\widetilde E_m=-ik_m\widetilde V_m}.
$$

Thus the written equations apply in either SI or normalized units, while substituting $\epsilon_0=1$ produces the values calculated in the code.

In Lecture 11, a finite-difference basis made the Laplacian a sparse matrix. Here, the Fourier basis makes the periodic Laplacian diagonal, so each transformed equation is solved by division.


### The special $k=0$ mode

The zero-frequency coefficient is proportional to the total normalized charge:

$$
\widetilde{\rho}(0)=\int_0^L\rho(x)\,dx.
$$

Integrating Gauss's law over a periodic domain gives

$$
E(L)-E(0)=\frac{1}{\epsilon_0}\int_0^L\rho(x)\,dx=0.
$$

The factor $1/\epsilon_0$ remains in the physical equation even though the code evaluates it with $\epsilon_0=1$.

Consequently, a periodic Poisson solution requires **net neutrality**, $\widetilde{\rho}(0)=0$. For the remaining zero modes:

- setting $\widetilde{V}(0)=0$ chooses the arbitrary additive constant, or gauge, of the potential;
- setting $\widetilde{E}(0)=0$ chooses zero mean electric field.

Subtracting the mean charge is equivalent to adding a uniform neutralizing background. That can be physically appropriate for a plasma model, but it should be an explicit modeling choice rather than a silent numerical repair.


### Worked check: one Fourier mode

Before writing a general solver, choose a unitless source with a known answer:

$$
\rho(x)=\rho_0\cos(k_mx),\qquad k_m=\frac{2\pi m}{L}.
$$

The exact fields, with $\epsilon_0$ retained explicitly, are

$$
V(x)=\frac{\rho_0}{\epsilon_0k_m^2}\cos(k_mx),
\qquad
E(x)=\frac{\rho_0}{\epsilon_0k_m}\sin(k_mx).
$$

The code below evaluates these expressions using the normalized value $\epsilon_0=1$.

**Discuss before running the code:** Which discrete Fourier coefficients should be nonzero? Why is the electric field shifted by one quarter wavelength relative to the charge density?


In [ ]:
# Define a periodic grid in normalized coordinates without duplicating x=L.
L = 1.0
N = 256
dx = L / N
x = np.arange(N) * dx

# Construct a unitless charge density containing one grid-resolved mode.
mode_number = 4
rho0 = 1.0  # normalized charge-density amplitude
k_mode = 2.0 * np.pi * mode_number / L
rho = rho0 * np.cos(k_mode * x)

# Transform the source and label its independent nonnegative wave numbers.
rho_k = rfft(rho)
k = 2.0 * np.pi * rfftfreq(N, d=dx)

# Choose V(k=0)=0 and solve every nonzero mode independently.
nonzero = k != 0.0
V_k = np.zeros_like(rho_k)
V_k[nonzero] = rho_k[nonzero] / (eps0 * k[nonzero]**2)

# Differentiate V in Fourier space to obtain the electric field.
E_k = -1j * k * V_k

# For even N, the Nyquist coefficient must remain real for a real output.
# Its sampled derivative cannot be represented on this same grid, so set it to zero.
E_k[-1] = 0.0

# Reconstruct the normalized real-space potential and electric field.
V_num = irfft(V_k)
E_num = irfft(E_k)

# Store exact and numerical values together for diagnostics and plotting.
single_mode_df = pd.DataFrame({
    "x": x,
    "rho": rho,
    "V_exact": rho0 * np.cos(k_mode * x) / (eps0 * k_mode**2),
    "V_fft": V_num,
    "E_exact": rho0 * np.sin(k_mode * x) / (eps0 * k_mode),
    "E_fft": E_num,
})

single_mode_df.head()


In [ ]:
# Plot directly from the DataFrame so the columns document each quantity.
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

axes[0].plot(single_mode_df["x"], single_mode_df["V_exact"],
             lw=3, alpha=0.55, label="exact")
axes[0].plot(single_mode_df["x"], single_mode_df["V_fft"],
             "--", lw=1.8, label="FFT solve")
axes[0].set_ylabel(r"Normalized potential $V$")
axes[0].legend()

axes[1].plot(single_mode_df["x"], single_mode_df["E_exact"],
             lw=3, alpha=0.55, label="exact")
axes[1].plot(single_mode_df["x"], single_mode_df["E_fft"],
             "--", lw=1.8, label="FFT solve")
axes[1].set_xlabel(r"Normalized position $x$")
axes[1].set_ylabel(r"Normalized field $E$")
axes[1].legend()

fig.suptitle("Single-mode validation of the periodic Poisson solver")
fig.tight_layout()
plt.show()

# Quantify agreement instead of relying only on visual overlap.
single_mode_errors = pd.DataFrame({
    "quantity": ["potential", "electric field"],
    "maximum_absolute_error": [
        np.max(np.abs(single_mode_df["V_fft"] - single_mode_df["V_exact"])),
        np.max(np.abs(single_mode_df["E_fft"] - single_mode_df["E_exact"])),
    ],
})
single_mode_errors


The FFT does not approximate the derivative by a local stencil. It represents the sampled function as a sum of grid-resolved Fourier modes and differentiates each mode exactly. For this smooth periodic test, the remaining disagreement should be close to floating-point roundoff.

## Part 3: In-class coding activity 2 - Build a reusable field solver

**Time: 14 minutes. Work in pairs.**

Complete `solve_periodic_poisson_fft`. The input DataFrame contains an even number of uniformly spaced normalized positions and a neutral, unitless charge density. The function should return:

- a real-space DataFrame containing $\rho(x)$, $V(x)$, and $E(x)$;
- a spectral DataFrame containing their independent, nonnegative-$k$ Fourier coefficients.

The argument `eps0` is retained to show the structure of the physical equation, but it has the normalized value `1.0` in every calculation here.

Before coding, identify the line at which each physical choice enters: periodic boundaries, net neutrality, the potential gauge, and the mean electric field.


In [ ]:
def solve_periodic_poisson_fft(charge_df, eps0):
    # Solve normalized 1D Poisson's equation for a neutral periodic source.
    # Read the normalized coordinate and unitless source from named columns.
    x_values = charge_df["x"].to_numpy()
    rho_values = charge_df["rho"].to_numpy()
    n_grid = len(charge_df)

    # The lecture uses even grids so rfft includes one Nyquist coefficient.
    if n_grid % 2 != 0:
        raise ValueError("This solver expects an even number of grid points")

    # TODO: Compute the uniform grid spacing from the position column.
    dx_local = ...

    # A periodic solution exists only for a net-neutral source.
    neutrality_tolerance = 1.0e-12 * np.max(np.abs(rho_values))
    if abs(np.mean(rho_values)) > neutrality_tolerance:
        raise ValueError("Periodic Poisson solve requires zero mean charge density")

    # TODO: Transform the real source with rfft.
    rho_k_local = ...
    # TODO: Use rfftfreq to construct the matching nonnegative k values.
    k_local = ...

    # TODO: Identify all stored modes except k=0.
    nonzero_local = ...

    # Choose V(k=0)=0 and solve Poisson's equation for every other mode.
    V_k_local = np.zeros_like(rho_k_local)
    # TODO: Fill the nonzero potential modes using Poisson's equation.
    V_k_local[nonzero_local] = ...

    # TODO: Compute E_k=-ik*V_k, including the zero-mean-field choice.
    E_k_local = ...

    # The derivative of the unpaired Nyquist mode is not represented on this grid.
    E_k_local[-1] = 0.0

    # TODO: Use irfft to reconstruct the real potential.
    V_local = ...
    # TODO: Use irfft to reconstruct the real electric field.
    E_local = ...

    # Return labeled tables that can be reused for plots and interpretation.
    solution_df = charge_df.copy()
    solution_df["V"] = V_local
    solution_df["E"] = E_local

    spectral_df = pd.DataFrame({
        "k": k_local,
        "rho_k": rho_k_local,
        "V_k": V_k_local,
        "E_k": E_k_local,
    })
    return solution_df, spectral_df


## Part 4: Apply the solver to a localized source

A localized plasma perturbation contains many Fourier modes. We will use a positive and a negative periodic Gaussian so that the total normalized charge is zero. The helper below computes the shortest signed distance around the periodic domain; without it, a Gaussian near a boundary would be cut rather than wrapped.


In [ ]:
def periodic_gaussian(x_values, center, width, domain_length):
    # Return a Gaussian using the shortest distance on a periodic domain.
    wrapped_distance = (
        (x_values - center + domain_length / 2.0) % domain_length
        - domain_length / 2.0
    )
    return np.exp(-0.5 * (wrapped_distance / width)**2)


# Build a smooth, localized, and explicitly neutral unitless source.
L_source = 1.0
N_source = 512
x_source = np.arange(N_source) * L_source / N_source
rho_amplitude = 1.0  # normalized charge-density amplitude

rho_source = rho_amplitude * (
    periodic_gaussian(x_source, 0.28 * L_source, 0.04 * L_source, L_source)
    - periodic_gaussian(x_source, 0.68 * L_source, 0.04 * L_source, L_source)
)

# Remove roundoff in the zero mode; physically this is the neutral source we chose.
rho_source -= np.mean(rho_source)
charge_df = pd.DataFrame({
    "x": x_source,
    "rho": rho_source,
})

# This cell assumes the activity function above has been completed correctly.
field_solution_df, field_spectrum_df = solve_periodic_poisson_fft(
    charge_df, eps0
)

charge_df.head()


In [ ]:
# Display the normalized source and the two fields returned by the solver.
fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

axes[0].plot(field_solution_df["x"], field_solution_df["rho"],
             color="tab:red")
axes[0].axhline(0.0, color="black", lw=0.8)
axes[0].set_ylabel(r"Charge density $\rho$")

axes[1].plot(field_solution_df["x"], field_solution_df["V"],
             color="tab:blue")
axes[1].set_ylabel(r"Potential $V$")

axes[2].plot(field_solution_df["x"], field_solution_df["E"],
             color="tab:green")
axes[2].axhline(0.0, color="black", lw=0.8)
axes[2].set_xlabel(r"Normalized position $x$")
axes[2].set_ylabel(r"Field $E$")

fig.suptitle("Normalized periodic fields from a neutral charge perturbation")
fig.tight_layout()
plt.show()


**Interpret the result:** Are the periodicity, zero mean electric field, and expected symmetries visible in the plotted solution? What features of the normalized charge distribution determine where $V(x)$ and $E(x)$ change most rapidly?


### Poisson's equation as a convolution

For a linear, translation-invariant periodic problem, the potential can be written as the response to every source point:

$$
V(x)=\int_0^L G(x-x')\rho(x')\,dx'.
$$

The function $G(x-x')$ is the periodic Green function. Because it depends only on the separation between the observation point and source point, the integral is a **convolution**. The convolution theorem converts this real-space integral into multiplication in Fourier space:

$$
\widetilde V(k)=\widetilde G(k)\widetilde\rho(k).
$$

Comparing this relation with the Fourier-space Poisson solution shows that, for nonzero $k$,

$$
\widetilde G(k)=\frac{1}{\epsilon_0k^2}.
$$

The implementation uses the normalized choice $\epsilon_0=1$, but the equation retains the physical constant.

The FFT solver therefore evaluates

$$
V(x)=\mathcal{F}^{-1}\!\left[
\widetilde G(k)\widetilde\rho(k)
\right].
$$

Because $G(x)$ and $\rho(x)$ are real, their one-sided `rfft` coefficients completely specify this product; `irfft` supplies the conjugate negative-$k$ terms during reconstruction. On the numerical grid this is a **cyclic discrete convolution**, consistent with periodic boundaries. A direct discrete convolution compares every source point with every field point and costs approximately $O(N^2)$; the FFT approach uses transforms and a multiplication, costing approximately $O(N\log N)$.

**Discuss:** Where did periodicity enter this calculation? What would go wrong if the desired physical system instead had conducting walls with fixed potential?


In [ ]:
# Construct the one-sided Green function in normalized Fourier space.
k_values = field_spectrum_df["k"].to_numpy()
green_k = np.zeros_like(k_values, dtype=complex)
nonzero_source = k_values != 0.0
green_k[nonzero_source] = 1.0 / (eps0 * k_values[nonzero_source]**2)

# Multiplication by G(k) is equivalent to cyclic convolution in real space.
rho_k_values = field_spectrum_df["rho_k"].to_numpy()
V_from_green = irfft(green_k * rho_k_values)

# Reconstruct the real response kernel and roll its origin to the plot center.
green_kernel = irfft(green_k)
green_df = pd.DataFrame({
    "separation": (np.arange(N_source) - N_source // 2) * (L_source / N_source),
    "discrete_kernel": np.roll(green_kernel, N_source // 2),
})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(green_df["separation"], green_df["discrete_kernel"])
axes[0].set_xlabel(r"Normalized periodic separation $x-x'$")
axes[0].set_ylabel("Discrete kernel weight")
axes[0].set_title("Periodic discrete response kernel")

axes[1].plot(field_solution_df["x"], field_solution_df["V"],
             lw=3, alpha=0.55, label="field solver")
axes[1].plot(field_solution_df["x"], V_from_green,
             "--", lw=1.8, label="Green-function product")
axes[1].set_xlabel(r"Normalized position $x$")
axes[1].set_ylabel(r"Normalized potential $V$")
axes[1].set_title("Two views of the same calculation")
axes[1].legend()

fig.tight_layout()
plt.show()


## Part 5: Where the field solver sits in a plasma calculation

Assignment 3 and this lecture share the same source-to-field structure: particles supply charge to a grid, a Poisson solve produces the grid field, and the field can then be interpolated back to particle positions:

$$
\{x_p,q_p\}
\xrightarrow{\text{deposit}}
\rho_j
\xrightarrow{\text{Poisson solve}}
V_j
\xrightarrow{\mathbf E=-\nabla V}
E_j
\xrightarrow{\text{interpolate}}
E(x_p)
\xrightarrow{\text{push}}
\{x_p,v_p\}.
$$

The middle step is not numerically identical in the two cases:

| Calculation | Boundary condition | Poisson solver |
|---|---|---|
| Assignment 3 | grounded Dirichlet walls in 2D | sparse finite-difference system from Lectures 11 and 12 |
| Lecture 14 | periodic domain in 1D | division by $k^2$ in the Fourier basis |

The particle-deposition and field-solve interfaces are conceptually shared, but the physical boundary conditions determine which solver is appropriate. A periodic FFT solver should not be substituted for the assignment's grounded-wall solver.

A complete repeated deposit-solve-interpolate-push cycle is an electrostatic particle-in-cell (PIC) algorithm. Production plasma codes add multiple dimensions, magnetic fields, relativistic particles, parallel domain decomposition, and careful filtering. Spectral electromagnetic solvers go further by evolving Maxwell's equations directly in Fourier space. The [WarpX documentation on explicit electromagnetic PIC](https://warpx.readthedocs.io/en/latest/theory/models_algorithms/explicit_em_pic.html) describes one such method, the pseudo-spectral analytical time-domain (PSATD) algorithm.

## Synthesis

The same real-valued numerical transform supports two conjugate-variable pairs:

1. **Time and angular frequency:** $t\leftrightarrow\omega$, used to identify temporal oscillations, amplitudes, and phases.
2. **Position and angular wave number:** $x\leftrightarrow k$, used to represent spatial structure and derivatives.

For periodic electrostatics,

$$
\underbrace{-\nabla^2 V=\rho/\epsilon_0}_{\text{position space}}
\quad\xrightarrow{\mathcal{F}}\quad
\underbrace{k^2\widetilde V=\widetilde\rho/\epsilon_0}_{k\text{-space}}.
$$

The equations retain $\epsilon_0$ explicitly, while the code uses the normalized value `eps0 = 1.0`.

`rfft` and `irfft` keep only the independent nonnegative-frequency coefficients while representing the same full DFT. The calculation is meaningful only when normalization, sampling, periodicity, the Nyquist mode, and the zero mode are treated consistently.


## References and further reading

- [SciPy FFT tutorial](https://docs.scipy.org/doc/scipy/tutorial/fft.html): conventions, discrete transforms, frequencies, and examples.
- [SciPy `rfft` documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.fft.rfft.html): the one-sided transform used throughout this notebook.
- [SciPy `rfftfreq` documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.fft.rfftfreq.html): ordering and units of nonnegative frequency bins.
- [WarpX: explicit electromagnetic PIC algorithms](https://warpx.readthedocs.io/en/latest/theory/models_algorithms/explicit_em_pic.html): production finite-difference and spectral Maxwell solvers.
- [Unit 3 numerical-methods resources](../resources/unit-3-numerical-methods.md): course references for ODEs, PDEs, sparse linear algebra, and Fourier methods.
- [Assignment 3 template](https://github.com/WM-PHYS690-Fall2026/assignment3-template): particle deposition and electrostatic field solving in the course project workflow.
